In [ ]:
import pandas as pd
import re
import json

# ============================
# 1. LOAD DATASET JSONL
# ============================
input_path = r"D:\textualentailment\tempo_entailment_dataset.jsonl"
print(f"📂 Membaca dataset dari: {input_path}")

# Tambahkan lines=True agar bisa baca format JSONL
df = pd.read_json(input_path, lines=True)

print(f"✅ Dataset berhasil dibaca. Jumlah baris awal: {len(df)}")
print("\n--- Preview Data Sebelum Preprocessing ---")
print(df.head())
print(f"\n📊 Kolom yang tersedia: {list(df.columns)}")

# Cek apakah kolom 'content' kosong
if 'content' in df.columns:
    content_kosong = df[df['content'].astype(str).str.strip() == '']
    print(f"\n🔍 Jumlah baris dengan content kosong: {len(content_kosong)}")
else:
    print("\n⚠️ Kolom 'content' tidak ditemukan dalam dataset.")

📂 Membaca dataset dari: D:\textualentailment\tempo_entailment_dataset.jsonl
✅ Dataset berhasil dibaca. Jumlah baris awal: 585107

--- Preview Data Sebelum Preprocessing ---
        doc_id                                              title  \
0  tempo_00001  Koleksi dan Panduan Pengunjung Grand Egyptian ...   
1  tempo_00001  Koleksi dan Panduan Pengunjung Grand Egyptian ...   
2  tempo_00001  Koleksi dan Panduan Pengunjung Grand Egyptian ...   
3  tempo_00001  Koleksi dan Panduan Pengunjung Grand Egyptian ...   
4  tempo_00001  Koleksi dan Panduan Pengunjung Grand Egyptian ...   

   paragraph_premise  paragraph_hypothesis  sentence_hypothesis  \
0                  1                     1                    1   
1                  1                     1                    2   
2                  1                     1                    3   
3                  1                     2                    1   
4                  1                     2                    2   

         

In [8]:
# ============================
# 2. TEXT CLEANING FUNCTION
# ============================
print("\n🧹 Step 2: Membersihkan teks...")

UNWANTED_PATTERNS = [
    # Pola-pola ini diubah dari ".*" (rakus) menjadi ":?" (opsional titik dua)
    # Ini akan menghapus frasa seperti "Pilihan Editor" atau "Pilihan Editor:"
    # tetapi TIDAK akan menghapus sisa teks di artikel.
    r"(?i)Pilihan Editor:?",
    r"(?i)BACA JUGA:?",
    r"(?i)Scroll ke bawah:?",
    r"(?i)Bagikan:?",
    r"(?i)Perbesar:?",
    r"(?i)Ringkasan Berita:?",
    r"(?i)Logo:?",
    r"(?i)Simak penjelasannya:?",
    r"(?i)Bergabung dengan grup Tempo:?",
    r"(?i)Komentar:?",
    r"(?i)Ikuti kami:?",
    r"(?i)Baca Juga:?",
    r"(?i)Klik di sini:?",
    r"(?i)Scroll:?",
    r"(?i)Dok\. Pribadi:?",

    # Pola ini diubah untuk hanya menghapus emoji-nya saja
    r"🔥",

    # Pola ini sudah Anda perbaiki dan sudah benar (non-greedy)
    r"(?i)TEMPO\.CO.*?[\-:]",

    # Pola-pola ini tidak memiliki ".*" dan sudah spesifik, jadi biarkan saja
    r"\d{1,2}\s\w+\s\d{4}\s\|\s\d{2}\.\d{2}\sWIB",
    r"^\s*(Politik|Ekonomi|Nasional|Dunia|Metro|Olah Raga|Teknologi|Bisnis|Hukum|Seleb|Lifestyle)\s*$",

    # Pola byline (Reporter/Redaktur/Editor) diperbaiki agar tidak terlalu rakus.
    # Ini akan menghapus "Reporter: John Doe" tapi berhenti di tanda "|" atau akhir kalimat.
    # Ini lebih aman daripada ".*Tempo.*" yang bisa menghapus seluruh artikel.
    r"(?i)Reporter\s*:.*?(?=(\||$|\.))",
    r"(?i)Redaktur\s*:.*?(?=(\||$|\.))",
    r"(?i)Editor\s*:.*?(?=(\||$|\.))"
]

def clean_text(text):
    if not isinstance(text, str) or pd.isna(text):
        return ""
    
    for pattern in UNWANTED_PATTERNS:
        text = re.sub(pattern, " ", text)
    
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    
    return text

# Bersihkan kolom premise, hypothesis, dan content
for col in ["premise", "hypothesis", "content"]:
    if col in df.columns:
        print(f"   🔄 Cleaning kolom: {col}")
        df[col] = df[col].astype(str).apply(clean_text)

print("✅ Text cleaning completed.")
print("-" * 80)


🧹 Step 2: Membersihkan teks...
   🔄 Cleaning kolom: premise
   🔄 Cleaning kolom: hypothesis
   🔄 Cleaning kolom: content
✅ Text cleaning completed.
--------------------------------------------------------------------------------


In [ ]:
import re
from tabulate import tabulate

# ============================
# FUNGSI POTONG CONTENT (1 KALIMAT)
# ============================
def short_content(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    first_sentence = re.split(r'(?<=[.!?])\s+', text.strip())[0]
    return first_sentence + " ..."

# Salinan untuk tampilan saja
df_view = df.copy()

if "content" in df_view.columns:
    df_view["content"] = df_view["content"].apply(short_content)

preview_cols = [
    "doc_id",
    "paragraph_premise",
    "paragraph_hypothesis",
    "sentence_hypothesis",
    "premise",
    "hypothesis",
    "content"
]

print("\nPreview 5 baris data (content 1 kalimat):\n")
print(
    tabulate(
        df_view[preview_cols].head(5),
        headers="keys",
        tablefmt="fancy_grid",
        showindex=False,
        maxcolwidths=[12, 8, 8, 8, 35, 35, 45]
    )
)1



Preview 5 baris data (content 1 kalimat):

╒═════════════╤═════════════════════╤════════════════════════╤═══════════════════════╤═════════════════════════════════════╤═════════════════════════════════════╤═══════════════════════════════════════════════╕
│ doc_id      │   paragraph_premise │   paragraph_hypothesis │   sentence_hypothesis │ premise                             │ hypothesis                          │ content                                       │
╞═════════════╪═════════════════════╪════════════════════════╪═══════════════════════╪═════════════════════════════════════╪═════════════════════════════════════╪═══════════════════════════════════════════════╡
│ tempo_00001 │                   1 │                      1 │                     1 │ Grand Egyptian Museum (GEM) dekat   │ MUSEUM Mesir Agung atau Grand       │ MUSEUM Mesir Agung atau Grand Egyptian Museum │
│             │                     │                        │                       │ Piramida Giza resmi buka 

In [10]:
# ============================
# 3. DROP DUPLICATES
# ============================
print("\nStep 3: Menghapus duplikat...")
before_drop = len(df)
df.drop_duplicates(subset=["premise", "hypothesis"], inplace=True, keep='first')
after_drop = len(df)

print(f"Duplikat dihapus: {before_drop - after_drop}")
print(f" Jumlah data unik sekarang: {after_drop}")
print("-" * 80)


Step 3: Menghapus duplikat...
Duplikat dihapus: 9422
 Jumlah data unik sekarang: 575685
--------------------------------------------------------------------------------


In [13]:
import json
from rich.console import Console
from rich.table import Table

console = Console()

input_path = r"D:\textualentailment\tempo_entailment_dataset.jsonl"

# ============================
# STATISTIK SEBELUM & SESUDAH
# ============================
before = {
    "rows": 0,
    "premise_sum": 0,
    "hypothesis_sum": 0,
    "content_sum": 0,
    "premise_min": float("inf"),
    "premise_max": 0,
    "hypothesis_min": float("inf"),
    "hypothesis_max": 0,
    "content_min": float("inf"),
    "content_max": 0,
}

after = {
    "rows": 0,
    "premise_sum": 0,
    "hypothesis_sum": 0,
    "content_sum": 0,
    "premise_min": float("inf"),
    "premise_max": 0,
    "hypothesis_min": float("inf"),
    "hypothesis_max": 0,
    "content_min": float("inf"),
    "content_max": 0,
}

seen = set()

# ============================
# STREAMING DATASET
# ============================
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)

        p = len(r.get("premise", "").split())
        h = len(r.get("hypothesis", "").split())
        c = len(r.get("content", "").split())

        # ---------
        # SEBELUM DEDUP
        # ---------
        before["rows"] += 1
        before["premise_sum"] += p
        before["hypothesis_sum"] += h
        before["content_sum"] += c

        before["premise_min"] = min(before["premise_min"], p)
        before["premise_max"] = max(before["premise_max"], p)

        before["hypothesis_min"] = min(before["hypothesis_min"], h)
        before["hypothesis_max"] = max(before["hypothesis_max"], h)

        before["content_min"] = min(before["content_min"], c)
        before["content_max"] = max(before["content_max"], c)

        # ---------
        # SESUDAH DEDUP
        # ---------
        key = (
            r.get("doc_id", ""),
            r.get("premise", ""),
            r.get("hypothesis", "")
        )

        if key in seen:
            continue

        seen.add(key)

        after["rows"] += 1
        after["premise_sum"] += p
        after["hypothesis_sum"] += h
        after["content_sum"] += c

        after["premise_min"] = min(after["premise_min"], p)
        after["premise_max"] = max(after["premise_max"], p)

        after["hypothesis_min"] = min(after["hypothesis_min"], h)
        after["hypothesis_max"] = max(after["hypothesis_max"], h)

        after["content_min"] = min(after["content_min"], c)
        after["content_max"] = max(after["content_max"], c)

# ============================
# TABEL PERBANDINGAN STATISTIK
# ============================
table = Table(title="Perbandingan Statistik Sebelum vs Sesudah Duplicate Removal")

table.add_column("Metrik")
table.add_column("Sebelum Dedup", justify="right")
table.add_column("Sesudah Dedup", justify="right")

table.add_row("Total baris dataset", f"{before['rows']:,}", f"{after['rows']:,}")

table.add_row(
    "Rata-rata panjang premise (kata)",
    f"{before['premise_sum']/before['rows']:.2f}",
    f"{after['premise_sum']/after['rows']:.2f}",
)
table.add_row(
    "Rata-rata panjang hypothesis (kata)",
    f"{before['hypothesis_sum']/before['rows']:.2f}",
    f"{after['hypothesis_sum']/after['rows']:.2f}",
)
table.add_row(
    "Rata-rata panjang content (kata)",
    f"{before['content_sum']/before['rows']:.2f}",
    f"{after['content_sum']/after['rows']:.2f}",
)

table.add_row(
    "Minimum premise (kata)",
    str(before["premise_min"]),
    str(after["premise_min"]),
)
table.add_row(
    "Maksimum premise (kata)",
    str(before["premise_max"]),
    str(after["premise_max"]),
)

table.add_row(
    "Minimum hypothesis (kata)",
    str(before["hypothesis_min"]),
    str(after["hypothesis_min"]),
)
table.add_row(
    "Maksimum hypothesis (kata)",
    str(before["hypothesis_max"]),
    str(after["hypothesis_max"]),
)

table.add_row(
    "Minimum content (kata)",
    str(before["content_min"]),
    str(after["content_min"]),
)
table.add_row(
    "Maksimum content (kata)",
    str(before["content_max"]),
    str(after["content_max"]),
)

console.print(table)


      Perbandingan Statistik Sebelum vs Sesudah Duplicate Removal      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Metrik                              ┃ Sebelum Dedup ┃ Sesudah Dedup ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Total baris dataset                 │       585,107 │       576,501 │
│ Rata-rata panjang premise (kata)    │         16.75 │         16.74 │
│ Rata-rata panjang hypothesis (kata) │         16.65 │         16.72 │
│ Rata-rata panjang content (kata)    │        496.13 │        486.90 │
│ Minimum premise (kata)              │             1 │             1 │
│ Maksimum premise (kata)             │            59 │            59 │
│ Minimum hypothesis (kata)           │             1 │             1 │
│ Maksimum hypothesis (kata)          │           633 │           633 │
│ Minimum content (kata)              │             1 │             1 │
│ Maksimum content (kata)             │          5519 │          5519 │
└─────────────────────────────────────┴───────────────┴───────────────┘

In [14]:
# ============================
# 5. SIMPAN KE JSONL
# ============================
output_jsonl = r"D:\textualentailment\tempo_dataset_cleaned_TERBARU.jsonl"

with open(output_jsonl, "w", encoding="utf-8") as f:
    for record in df.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"✅ Dataset JSONL bersih disimpan ke: {output_jsonl}")
print(f"📦 Total data akhir: {len(df)} baris")
print("\n✨ Preprocessing & Cleaning Selesai!")

✅ Dataset JSONL bersih disimpan ke: D:\textualentailment\tempo_dataset_cleaned_TERBARU.jsonl
📦 Total data akhir: 575685 baris

✨ Preprocessing & Cleaning Selesai!
